In [ ]:
import logging


# capture all loggers after importing modules
all_loggers = [logging.getLogger(name) for name in logging.root.manager.loggerDict]
print(f"Total loggers found: {len(all_loggers)}")
for logger in all_loggers:
    print(f"Logger: {logger.name} - Level: {logger.level}")

In [ ]:
import os
import sys


sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

In [ ]:
from config import get_settings
from utils.logging import get_logger


settings = get_settings()
logger = get_logger(__name__)

In [ ]:
page_id = "2a54eaab90d28183ae14c83ea3be31d9"
fact_id = "2a54eaab90d281ca85b4c757a7afb1c1"
template_slug = "terras-gerais"

In [ ]:
from services.data.plot_data import PlotDataExtractor
from services.html.render import HTMLRenderer
from services.notion.notion_service import NotionService
from services.report.generate_report import ReportGenerator


generator = ReportGenerator()
notion_service = NotionService()
plot_extractor = PlotDataExtractor()
html_renderer = HTMLRenderer()

## generator.report_with_template

In [ ]:
fact_ds_id, talhoes_ds_id = notion_service.get_fact_and_talhoes_data_sources()
print(fact_ds_id, talhoes_ds_id)

In [ ]:
fact_data = await notion_service.async_query_fact_by_page_id(fact_ds_id, page_id)
print(fact_data)

In [ ]:
fact_item = fact_data[0]
farm_name = fact_item.get("nome_fazenda")
print(fact_item)
print(farm_name)

In [ ]:
talhoes_data = notion_service.query_talhoes_by_farm_ids(talhoes_ds_id, fact_item.get("farm"))
print(talhoes_data)

In [ ]:
page_data = await notion_service.async_get_page(page_id)
print(page_data)

In [ ]:
plots_with_images = await plot_extractor.extract_plots_data(page_id)
print(plots_with_images)

In [ ]:
plots_with_images

In [ ]:
enriched_plots = notion_service.merge_plot_data(talhoes_data, plots_with_images)
print(enriched_plots)

In [ ]:
report_data = generator._build_report_model(
                fact_item=fact_item,
                page_data=page_data,
                plots=enriched_plots,
                farm_name=farm_name,
                talhoes_data=talhoes_data,
            )
print(report_data)

In [ ]:
html_content = await html_renderer.render_template_slug(
    template_slug, report_data
)
print(html_content)